In [1]:
import os
import sys
from dataclasses import dataclass

import optuna
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../.."))

import src.utils.run_optuna as op
from src.utils.get_objective import get_objective

In [2]:
# === Configuration === (you edit here) ===
load_dotenv(dotenv_path="../../.env")


@dataclass
class Config:
    # Data / CV
    model_name: str = "mlp"
    data_id: str = "033"
    n_fold: int = 5
    seed: int = 42
    fold_idx: int = 0

    # Optuna
    n_trials: int = 1
    direction: str = "maximize"
    sampler: str = "tpe"  # tpe / random
    pruner: str = "median"  # median / none

    # Storage
    storage: str = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"


cfg = Config()

opts = {
    "earlys_stopping_rounds": 5,
    "max_epochs": 10,
    "min_epochs": 4,
}

# W&B
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc
wandb: Currently logged in as: kaitookano (kaitookano-waseda-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# === Build & Run (frozen) ===
# sampler / pruner factory
def build_sampler(name, seed):
    if name == "tpe":
        return optuna.samplers.TPESampler(n_startup_trials=15, seed=seed)
    elif name == "random":
        return optuna.samplers.RandomSampler(seed=seed)
    else:
        raise ValueError(f"unknown sampler: {name}")


def build_pruner(name):
    if name == "median":
        return optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1000)
    elif name == "none":
        return optuna.pruners.NopPruner()
    else:
        raise ValueError(f"unknown pruner: {name}")


create_objective = get_objective(cfg.model_name)
objective = create_objective(
    cfg.data_id,
    seed=cfg.seed,
    n_fold=cfg.n_fold,
    fold_idx=cfg.fold_idx,
    wandb_project=wandb_project,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    opts=opts
)

sampler = build_sampler(cfg.sampler, cfg.seed)
pruner = build_pruner(cfg.pruner)

op.run_optuna_search(
    objective,
    n_trials=cfg.n_trials,
    n_jobs=1,
    direction=cfg.direction,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    storage=cfg.storage,
    sampler=sampler,
    pruner=pruner
)

[I 2025-09-26 14:33:19,922] Using an existing study with name 'mlp-033' instead of creating a new one.


  0%|          | 0/1 [00:00<?, ?it/s]

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Fold Col: 5fold-s42
Free CPU Mem: 16.68 GB
Free GPU Mem: 6.61 GB
Epoch 1: Train Logloss = 0.13641, Val Logloss = 0.13525
New best model saved at epoch 1, Logloss: 0.13525
Epoch 2: Train Logloss = 0.13342, Val Logloss = 0.13302
New best model saved at epoch 2, Logloss: 0.13302
Epoch 3: Train Logloss = 0.13242, Val Logloss = 0.13250
New best model saved at epoch 3, Logloss: 0.13250
Epoch 4: Train Logloss = 0.13137, Val Logloss = 0.13198
New best model saved at epoch 4, Logloss: 0.13198
Epoch 5: Train Logloss = 0.13056, Val Logloss = 0.13188
New best model saved at epoch 5, Logloss: 0.13188
Epoch 6: Train Logloss = 0.12983, Val Logloss = 0.13173
New best model saved at epoch 6, Logloss: 0.13173
Epoch 7: Train Logloss = 0.12892, Val Logloss = 0.13168
New best model saved at epoch 7, Logloss: 0.13168
Epoch 8: Train Logloss = 0.12816, Val Logloss = 0.13172
Epoch 9: Train Logloss = 0.12719, Val Logloss = 0.13162
New best model saved at epoch 9, Logloss: 0.13162
Epoch 10: Train Logloss = 0.126

epoch_f1,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
meta/f1/lr,█████▇▇▇▆▆▆▅▅▄▄▃▃▂▂▁
train/f1/auc,▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▇▇█
train/f1/loss,█▇▇▇▇▆▆▆▆▅▅▅▅▄▄▃▃▂▂▁
valid/f1/auc,▄▆▇▇▇██████▇▇▆▆▅▄▃▂▁
valid/f1/loss,▃▂▁▁▁▁▁▁▁▁▁▁▂▂▃▄▄▅▆█
auc_f1,0.97382
epoch_f1,19
loss_f1,0.13162
meta/f1/lr,0.00139
runtime_f1,1.68557


[I 2025-09-26 14:35:13,975] Trial 4 finished with value: 0.9738210324935945 and parameters: {'num_layers': 2, 'hidden_dim1': 992, 'hidden_dim2': 768, 'hidden_dim3': -1, 'hidden_dim4': -1, 'batch_size': 864, 'lr': 0.0020513382630874496, 'eta_min': 0.00014321698289111514, 'dropout_rate': 0.1, 'activation': 'ReLU'}. Best is trial 2 with value: 0.9744404354691245.
✅ Message sent.
